# Модуль-ноутбук: `inference`

Кандидат → {рекомендація, ймовірність, фактори, схожі відео}. Лише pre-post вхід, ніколи не падає. Використовує збережений `models/model.joblib` (натренуй у `05_train.ipynb`).

**Залежності:** `%run` 01_features.ipynb

In [ ]:
%run 01_features.ipynb

In [ ]:
"""Inference: a not-yet-posted candidate -> {recommendation, probability, factors, evidence}.

Pre-post inputs ONLY. We never fetch or accept post-hoc engagement counts — that is the
leakage trap. `parse_tiktok_url` extracts only the handle/id from a pasted URL (no network).

Public API:
    recommend(caption, duration, when, creator=None) -> dict
"""

import math
import re
from datetime import datetime, timezone
from functools import lru_cache

import joblib
import numpy as np
import pandas as pd


# ----------------------------------------------------------------------------- loading

In [ ]:
@lru_cache(maxsize=1)
def load_bundle(path: str | None = None) -> dict:
    p = path or config.MODEL_PATH
    bundle = joblib.load(p)
    bundle["nn_meta"] = pd.DataFrame(bundle["nn_meta"])
    return bundle

In [ ]:
def is_trained() -> bool:
    return config.MODEL_PATH.exists()

In [ ]:
@lru_cache(maxsize=1)
def _creator_medians() -> dict:
    import json
    try:
        meta = json.loads(config.METADATA_PATH.read_text())
        return meta["creator_thresholds"]["per_creator"]
    except Exception:
        return {}

In [ ]:
def _creator_median_er(creator: str):
    return _creator_medians().get(creator)


# ----------------------------------------------------------------------------- input hygiene
_URL_RE = re.compile(r"tiktok\.com/@([\w.\-]+)(?:/video/(\d+))?", re.IGNORECASE)
POSTHOC = set(config.POSTHOC_COLS)

In [ ]:
def parse_tiktok_url(url: str) -> dict:
    """Extract ONLY @handle and video id from a URL string. Never fetches engagement."""
    m = _URL_RE.search(str(url) if url is not None else "")
    if not m:
        return {}
    return {"creator": m.group(1), "video_id": m.group(2)}

In [ ]:
def assert_no_posthoc(payload: dict) -> None:
    """Defensive guard: refuse any attempt to pass post-hoc metrics as input."""
    leaked = POSTHOC.intersection(payload or {})
    if leaked:
        raise ValueError(f"post-hoc metrics are not allowed as inputs (leakage): {sorted(leaked)}")

In [ ]:
def _validate_epoch(e: float) -> tuple[float, list[str]]:
    """Reject non-finite / out-of-range epochs so downstream timestamp math never raises."""
    try:
        e = float(e)
    except (TypeError, ValueError, OverflowError):
        return float("nan"), ["Unrecognized post time — time features treated as unknown."]
    if not math.isfinite(e) or e > config.MAX_PLAUSIBLE_EPOCH or e < 0:
        return float("nan"), ["Post time out of plausible range — time features treated as unknown."]
    return e, []

In [ ]:
def _to_epoch(when) -> tuple[float, list[str]]:
    """Normalize various 'when' inputs to a UTC epoch. Returns (epoch_or_nan, warnings).
    Never raises: any unparseable/out-of-range value degrades to NaN + a warning."""
    if when is None or (isinstance(when, float) and math.isnan(when)) or when == "":
        return float("nan"), ["No post time given — time-of-day features are treated as unknown."]
    if isinstance(when, bool):  # bool is an int subclass — reject before numeric handling
        return float("nan"), ["Unrecognized post time — time features treated as unknown."]
    if isinstance(when, (int, float)):
        return _validate_epoch(when)
    if isinstance(when, datetime):
        try:
            dt = when if when.tzinfo else when.replace(tzinfo=timezone.utc)
            return _validate_epoch(dt.timestamp())
        except (OverflowError, ValueError, OSError):
            return float("nan"), ["Post time out of plausible range — time features treated as unknown."]
    if isinstance(when, str):
        try:
            dt = datetime.fromisoformat(when)
            dt = dt if dt.tzinfo else dt.replace(tzinfo=timezone.utc)
            return _validate_epoch(dt.timestamp())
        except (ValueError, TypeError, OverflowError):
            pass
        try:  # last resort: pandas can parse many formats
            dt = pd.to_datetime(when, utc=True).to_pydatetime()
            return _validate_epoch(dt.timestamp())
        except Exception:
            return float("nan"), [f"Could not parse post time {when!r} — time features treated as unknown."]
    return float("nan"), ["Unrecognized post time — time features treated as unknown."]

In [ ]:
def _clean_duration(duration) -> tuple[float, list[str]]:
    warns: list[str] = []
    if isinstance(duration, bool):  # bool is an int subclass — not a real duration
        return float("nan"), ["Duration non-numeric — treated as unknown."]
    try:
        d = float(duration)
    except (TypeError, ValueError, OverflowError):
        return float("nan"), ["Duration missing or non-numeric — treated as unknown."]
    if not math.isfinite(d):
        return float("nan"), ["Duration non-finite — treated as unknown."]
    if math.isnan(d):
        return float("nan"), ["Duration missing — treated as unknown."]
    if d <= 0:
        return float("nan"), ["Duration ≤ 0 — treated as unknown."]
    if d > config.DURATION_CLIP[1]:
        warns.append(f"Duration {d:.0f}s is long; clipped to {config.DURATION_CLIP[1]:.0f}s for the model.")
    return d, warns


# ----------------------------------------------------------------------------- explanations

In [ ]:
def _linear_factors(bundle, X: pd.DataFrame, top_k: int) -> list[dict]:
    """Exact signed log-odds contributions from the logistic pipeline: coef * scaled_x."""
    pipe = bundle["model"]
    z = pipe[:-1].transform(X)[0]                 # impute + scale
    coef = pipe[-1].coef_[0]
    contrib = coef * z
    order = np.argsort(np.abs(contrib))[::-1][:top_k]
    return _format_factors(bundle, X, order, contrib)

In [ ]:
def _agnostic_factors(bundle, X: pd.DataFrame, top_k: int) -> list[dict]:
    """Model-agnostic local sensitivity: Δ(prob) when a feature is reset to its median."""
    model = bundle["model"]
    base_p = float(model.predict_proba(X)[0, 1])
    med = bundle["feature_medians"]
    contrib = np.zeros(len(features.FEATURE_COLUMNS))
    for i, f in enumerate(features.FEATURE_COLUMNS):
        Xm = X.copy()
        Xm.iloc[0, i] = med.get(f, np.nan)
        contrib[i] = base_p - float(model.predict_proba(Xm)[0, 1])
    order = np.argsort(np.abs(contrib))[::-1][:top_k]
    return _format_factors(bundle, X, order, contrib)

In [ ]:
def _format_factors(bundle, X, order, contrib) -> list[dict]:
    out = []
    for i in order:
        f = features.FEATURE_COLUMNS[i]
        if abs(contrib[i]) < 1e-9:
            continue
        out.append({
            "feature": f,
            "label": features.FEATURE_LABELS.get(f, f),
            "value": None if pd.isna(X.iloc[0, i]) else round(float(X.iloc[0, i]), 3),
            "contribution": round(float(contrib[i]), 4),
            "direction": "increases" if contrib[i] > 0 else "decreases",
        })
    return out

In [ ]:
def _similar_videos(bundle, X: pd.DataFrame, creator: str | None, k: int) -> list[dict]:
    """Nearest training videos in standardized feature space — real-outcome evidence."""
    meta: pd.DataFrame = bundle["nn_meta"]
    matrix = bundle["nn_matrix"]
    Xf = X.fillna(pd.Series(bundle["feature_medians"]))
    z = bundle["nn_scaler"].transform(Xf)[0]
    mask = np.ones(len(meta), dtype=bool)
    if creator and (meta["creator"] == creator).any():
        mask = (meta["creator"] == creator).to_numpy()  # prefer same-creator evidence
    dist = np.linalg.norm(matrix[mask] - z, axis=1)
    idx = np.where(mask)[0][np.argsort(dist)[:k]]
    rows = []
    for j in idx:
        r = meta.iloc[j]
        rows.append({
            "creator": str(r["creator"]),
            "caption": (str(r["description"])[:120] or "(no caption)"),
            "duration": None if pd.isna(r["duration"]) else int(r["duration"]),
            "engagement_rate": None if pd.isna(r["engagement_rate"]) else round(float(r["engagement_rate"]), 4),
            "outcome": "over-performed" if int(r["label"]) == 1 else "under-performed",
        })
    return rows


# ----------------------------------------------------------------------------- the contract
CAN_KNOW = [
    "Caption text patterns (length, hashtags, mentions, emoji, CTA, tone).",
    "Planned duration and its bucket.",
    "Planned posting time-of-day / weekday (assumed UTC).",
]
CANNOT_KNOW = [
    "The video and audio themselves — the dominant driver of TikTok performance.",
    "Trend timing, sound virality, and how the For-You algorithm will distribute it.",
    "Audience mood on the day, current events, and follower growth since training.",
    "Anything post-publish (views, likes) — those define the target and are never inputs.",
]

In [ ]:
def recommend(caption=None, duration=None, when=None, creator=None,
              top_k: int = 5, n_similar: int = 3, bundle_path: str | None = None) -> dict:
    """Score a not-yet-posted candidate. Always returns a dict; never raises on bad input."""
    if not is_trained() and bundle_path is None:
        return {"error": "Model not trained yet. Run `notebooks/05_train.ipynb` first."}

    warnings: list[str] = []
    caption = "" if caption is None else str(caption)
    if not caption.strip():
        warnings.append("Empty caption — captions carry signal here, so this lowers confidence.")
    if len(caption) > 2200:
        caption = caption[:2200]
        warnings.append("Caption exceeds TikTok's 2200-char limit; truncated.")

    dur, w = _clean_duration(duration); warnings += w
    epoch, w = _to_epoch(when); warnings += w
    creator = (str(creator).lstrip("@").strip() or None) if creator else None

    bundle = load_bundle(bundle_path)
    raw = pd.DataFrame([{
        "description": caption,
        "duration": dur,
        config.TIME_COL: epoch,
        config.CREATOR_COL: creator,
    }])
    X = features.engineer_features(raw)[bundle["feature_columns"]]
    features.assert_no_leakage(X)

    p_raw = float(bundle["model"].predict_proba(X)[0, 1])
    # Product guardrail: never be confident when basic inputs are missing/invalid.
    # Each missing critical field shrinks the calibrated margin (p-0.5) toward 0.5,
    # pulling incomplete candidates toward "Unsure" instead of a spurious confident call.
    missing_fields = [name for name, miss in (
        ("caption", not caption.strip()),
        ("duration", math.isnan(dur)),
        ("post time", math.isnan(epoch) or epoch < config.MIN_PLAUSIBLE_EPOCH),
    ) if miss]
    shrink = 0.5 ** len(missing_fields)
    p = 0.5 + (p_raw - 0.5) * shrink
    if missing_fields:
        warnings.append(
            f"Confidence reduced — {', '.join(missing_fields)} missing/invalid and not "
            f"scored: raw score {p_raw:.0%} → adjusted {p:.0%}.")

    t_low = bundle["decision_band"]["t_low"]
    t_high = bundle["decision_band"]["t_high"]
    if p >= t_high:
        rec, reason = "Post", f"Calibrated score {p:.0%} ≥ post threshold {t_high:.0%}."
    elif p <= t_low:
        rec, reason = "Do not post", f"Calibrated score {p:.0%} ≤ skip threshold {t_low:.0%}."
    else:
        rec, reason = "Unsure", f"Score {p:.0%} is inside the abstention band [{t_low:.0%}, {t_high:.0%}]."
    if missing_fields:
        reason += f" ({', '.join(missing_fields)} not provided, so confidence was reduced.)"

    factors = (_linear_factors if bundle["explainer"] == "linear" else _agnostic_factors)(bundle, X, top_k)
    similar = _similar_videos(bundle, X, creator, n_similar)

    creator_note = None
    if creator:
        # Surface the creator's historical median ER if known (context only — NOT a feature).
        cm = _creator_median_er(creator)
        creator_note = (
            f"@{creator} is a known creator; 'success' here means beating their historical "
            f"median engagement rate (~{cm:.1%})." if cm is not None
            else f"@{creator} is not in the training set; using the global engagement norm."
        )

    return {
        "recommendation": rec,
        "probability": round(p, 4),
        "decision_reason": reason,
        "thresholds": {"t_low": t_low, "t_high": t_high},
        "factors": factors,
        "similar_videos": similar,
        "creator_note": creator_note,
        "warnings": warnings,
        "model_can_know": CAN_KNOW,
        "model_cannot_know": CANNOT_KNOW,
        "label_meaning": "Probability that this video beats its creator's typical "
                         "(median) engagement rate — i.e. resonates better than usual.",
    }

In [ ]:
from types import SimpleNamespace
inference = SimpleNamespace(
    recommend=recommend,
    parse_tiktok_url=parse_tiktok_url,
    assert_no_posthoc=assert_no_posthoc,
    is_trained=is_trained,
    load_bundle=load_bundle,
    CAN_KNOW=CAN_KNOW,
    CANNOT_KNOW=CANNOT_KNOW,
)

### Перевірка / демо

In [ ]:
if not inference.is_trained():
    print('Спершу запусти 05_train.ipynb (немає models/model.joblib)')
else:
    r = inference.recommend(caption='POV: трюк нарешті вдався 🔥 #fyp',
                            duration=18, when='2025-03-01T19:30:00', creator='zachking')
    print(r['recommendation'], r['probability'])
    print('фактори:', [f['label'] for f in r['factors'][:3]])